In [ ]:
pip install faker

In [33]:
from faker import Faker
import pandas as pd
import random

# Initialiser Faker

In [34]:
fake = Faker()

# Générer des données pour la table Joueurs

In [35]:
from datetime import datetime

In [36]:
# Fonction pour calculer l'âge à partir de la date de naissance
def calculer_age(date_naissance):
    aujourdhui = datetime.today()
    return aujourdhui.year - date_naissance.year - ((aujourdhui.month, aujourdhui.day) < (date_naissance.month, date_naissance.day))

In [37]:
# Fonction pour assigner la catégorie en fonction de l'âge
def assigner_categorie(age):
    if age == 14:
        return random.choice(["Minime B", "Minime A"])
    elif age == 15:
        return random.choice(["Minime A", "Cadet B"])
    elif age == 16:
        return random.choice(["Cadet B", "Cadet A", "Junior"])
    elif age == 17:
        # Un seul joueur peut être Senior à 17 ans
        if random.random() < 0.05:  # 5% de chance d'être Senior
            return "Senior"
        else:
            return random.choice(["Cadet A", "Junior", "Elite"])
    elif age == 18:
        return random.choice(["Junior", "Elite", "Senior"])
    elif age == 19 or age == 20:
        return random.choice(["Elite", "Senior"])
    else:
        return "Senior"

In [38]:
# Fonction pour générer une taille réaliste en fonction de l'âge
def generer_taille(age):
    if age == 14:
        return round(random.uniform(150, 170), 1)  # Taille en cm
    elif age == 15:
        return round(random.uniform(155, 175), 1)
    elif age == 16:
        return round(random.uniform(160, 180), 1)
    elif age == 17:
        return round(random.uniform(165, 185), 1)
    elif age == 18:
        return round(random.uniform(170, 190), 1)
    elif age >= 19:
        return round(random.uniform(175, 195), 1)

In [39]:
# Fonction pour générer un poids réaliste en fonction de l'âge et de la taille
def generer_poids(age, taille):
    if age == 14:
        return round(random.uniform(45, 65), 1)  # Poids en kg
    elif age == 15:
        return round(random.uniform(50, 70), 1)
    elif age == 16:
        return round(random.uniform(55, 75), 1)
    elif age == 17:
        return round(random.uniform(60, 80), 1)
    elif age == 18:
        return round(random.uniform(65, 85), 1)
    elif age >= 19:
        return round(random.uniform(70, 90), 1)


In [40]:
# Fonction pour calculer l'IMC
def calculer_imc(poids, taille):
    return round(poids / ((taille / 100) ** 2), 1)  # IMC en kg/m²

In [41]:
# Générer des données pour la table Joueurs
data_joueurs = []
categories_count = {"Minime B": 0, "Minime A": 0, "Cadet B": 0, "Cadet A": 0, "Junior": 0, "Elite": 0, "Senior": 0}

# Liste des âges possibles
ages = [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]

# Nombre minimum et maximum de joueurs par catégorie
min_joueurs_par_categorie = 20
max_joueurs_par_categorie = 25

# Répartition des positions par catégorie
positions_par_categorie = {
    "Gardien": 3,
    "Défenseur": 5,
    "Milieu": 5,
    "Attaquant": 5
}

# Générer des joueurs jusqu'à ce que toutes les catégories aient entre 20 et 25 joueurs
while any(count < min_joueurs_par_categorie for count in categories_count.values()):
    age = random.choice(ages)
    categorie = assigner_categorie(age)
    
    # Vérifier que la catégorie n'a pas déjà 25 joueurs
    if categories_count[categorie] < max_joueurs_par_categorie:
        date_naissance = fake.date_of_birth(minimum_age=age, maximum_age=age)
        taille = generer_taille(age)
        poids = generer_poids(age, taille)
        imc = calculer_imc(poids, taille)
        
        # Assigner une position en fonction des besoins de la catégorie
        if categories_count[categorie] < 3:
            position = "Gardien"
        else:
            # Répartition équilibrée entre Défenseur, Milieu et Attaquant
            positions_disponibles = ["Défenseur", "Milieu", "Attaquant"]
            position = random.choice(positions_disponibles)
        
        data_joueurs.append({
            "ID_Joueur": fake.unique.random_int(min=1, max=1000),
            "Nom": fake.last_name(),
            "Prénom": fake.first_name(),
            "Date_Naissance": date_naissance,
            "Num_Licence": fake.unique.bothify(text="??####"),
            "Date_Adhésion": fake.date_between(start_date="-5y", end_date="today"),
            "Pied_Dominant": random.choice(["Gauche", "Droit", "Ambidextre"]),
            "Position": position,
            "Taille": taille,  # Taille en cm
            "Poids": poids,    # Poids en kg
            "IMC": imc,        # IMC en kg/m²
            "Catégorie": categorie,
            "Aspect_Général": round(random.uniform(1, 5), 1)  # Échelle de 1 à 5
        })
        categories_count[categorie] += 1

# Convertir en DataFrame
df_joueurs = pd.DataFrame(data_joueurs)

In [42]:
# Afficher la répartition des catégories
print("Répartition des catégories :")
print(df_joueurs["Catégorie"].value_counts())

Répartition des catégories :
Senior      25
Elite       25
Cadet B     25
Minime A    25
Junior      25
Cadet A     24
Minime B    20
Name: Catégorie, dtype: int64


In [43]:
# Exporter en CSV
df_joueurs.to_csv("joueurs.csv", index=False)

# Générer des données pour la table Matchs

In [44]:
from datetime import datetime, timedelta
import pandas as pd
import random

In [45]:

df_joueurs = pd.read_csv("joueurs.csv")  # Fichier contenant ID_Joueur, Catégorie, Poste


In [46]:
# Définition des saisons sportives
saisons = [
    ("2022-2023", datetime(2022, 8, 21), datetime(2023, 6, 30)),
    ("2023-2024", datetime(2023, 8, 21), datetime(2024, 6, 30)),
    ("2024-2025", datetime(2024, 8, 21), datetime(2025, 6, 30)),
]

In [47]:
# Scénarios possibles pour les matchs
scenarios_matchs = [
    {"Ligue": 30, "Coupe": 5, "Amical": 10},  # Gagné la coupe
    {"Ligue": 30, "Coupe": 4, "Amical": 7},   # Perdu en demi-finale
    {"Ligue": 30, "Coupe": 2, "Amical": 5},   # Éliminé tôt
]

In [48]:
# Assigner un scénario aléatoire à chaque catégorie
categories = df_joueurs["Catégorie"].unique()
repartition_categories = {}
scenarios_melanges = random.sample(scenarios_matchs, len(scenarios_matchs))

for i, categorie in enumerate(categories):
    repartition_categories[categorie] = scenarios_melanges[i % len(scenarios_melanges)]


In [49]:
# Fonction pour générer les dates des matchs
def generer_dates_matchs(debut_saison, fin_saison, nb_matchs):
    dates = []
    while len(dates) < nb_matchs:
        date_match = debut_saison + timedelta(days=random.randint(0, (fin_saison - debut_saison).days))
        if date_match.weekday() in [5, 6]:  # Jouer uniquement le samedi et dimanche
            dates.append(date_match)
    return sorted(dates)

In [50]:
import random

def assigner_titulaires_et_temps(joueurs_categorie):
    # Séparer les gardiens des autres joueurs
    gardiens = [j for j in joueurs_categorie if j["Position"] == "Gardien"]
    joueurs_sans_gardien = [j for j in joueurs_categorie if j["Position"] != "Gardien"]

    if len(gardiens) < 1:
        raise ValueError("Aucun gardien disponible dans cette catégorie.")

    # Sélection du gardien titulaire (il joue toujours 90 minutes)
    gardien_titulaire = random.choice(gardiens)
    gardien_titulaire["Temps_Jeu"] = 90

    # Sélection des autres titulaires (10 joueurs)
    autres_titulaires = random.sample(joueurs_sans_gardien, 10)

    # Vérifier qu'il y a au moins 3 défenseurs parmi les titulaires
    defenseurs = [j for j in autres_titulaires if j["Position"] == "Défenseur"]
    if len(defenseurs) < 3:
        defenseurs_supp = random.sample([j for j in joueurs_sans_gardien if j["Position"] == "Défenseur" and j not in autres_titulaires], 3 - len(defenseurs))
        for d in defenseurs_supp:
            autres_titulaires.remove(random.choice(autres_titulaires))  # Retirer un autre joueur au hasard
            autres_titulaires.append(d)

    # Liste finale des titulaires
    titulaires = [gardien_titulaire] + autres_titulaires

    # Donner le temps de jeu aux titulaires
    # 6 joueurs jouent 90 min (gardien + 5 autres joueurs aléatoires)
    joueurs_90_min = random.sample(titulaires, 5) + [gardien_titulaire]
    for j in joueurs_90_min:
        j["Temps_Jeu"] = 90

    # Les 5 autres titulaires sont remplacés en deuxième mi-temps (45min ou plus)
    remplaçants = [j for j in joueurs_categorie if j not in titulaires]
    joueurs_remplaces = [j for j in titulaires if j not in joueurs_90_min]

    # Sélection des 5 remplaçants
    if len(remplaçants) < 5:
        raise ValueError("Pas assez de joueurs pour effectuer les changements.")

    remplaçants_entrants = random.sample(remplaçants, 5)

    # Donner le temps de jeu aux remplaçants et aux titulaires remplacés
    for i in range(5):
        titulaire_sortant = joueurs_remplaces[i]
        remplaçant = remplaçants_entrants[i]
        
        # Remplacement entre la 45e et la 75e minute
        minute_remplacement = random.randint(45, 75)
        
        titulaire_sortant["Temps_Jeu"] = minute_remplacement
        remplaçant["Temps_Jeu"] = 90 - minute_remplacement

    # Tous les autres joueurs restent sur le banc avec 0 minute
    for j in joueurs_categorie:
        if "Temps_Jeu" not in j:
            j["Temps_Jeu"] = 0

    return joueurs_categorie, titulaires


In [51]:
# Génération des matchs
data_matchs = []
id_match_counter = 1

for saison, debut_saison, fin_saison in saisons:
    for categorie, joueurs_categorie in df_joueurs.groupby("Catégorie"):
        joueurs_categorie = joueurs_categorie.to_dict('records')
        scenario = repartition_categories[categorie]

        # Générer les dates des matchs pour chaque type
        dates_ligue = generer_dates_matchs(debut_saison, fin_saison, scenario["Ligue"])
        dates_coupe = generer_dates_matchs(debut_saison, fin_saison, scenario["Coupe"])
        dates_amical = generer_dates_matchs(debut_saison, fin_saison, scenario["Amical"])

        for date_match, competition in zip(dates_ligue + dates_coupe + dates_amical, ["Ligue"] * len(dates_ligue) + ["Coupe"] * len(dates_coupe) + ["Amical"] * len(dates_amical)):
            try:
                joueurs_categorie, titulaires = assigner_titulaires_et_temps(joueurs_categorie)
                total_buts_match = random.randint(0, 5)
                buts_restants = total_buts_match

                for joueur in joueurs_categorie:
                    carton_jaune, carton_rouge, fautes_commisses, buts = 0, 0, 0, 0
                    if joueur["Temps_Jeu"] > 0:
                        if joueur["Position"] in ["Défenseur", "Milieu"]:
                            fautes_commisses = random.randint(0, 5)
                            carton_jaune = 1 if fautes_commisses >= 2 else 0
                            carton_rouge = 1 if random.random() < 0.1 else 0
                        elif joueur["Position"] in ["Attaquant", "Gardien"]:
                            fautes_commisses = random.randint(0, 2) if random.random() < 0.3 else 0
                            carton_jaune = 1 if fautes_commisses >= 2 else 0

                        if buts_restants > 0:
                            if joueur["Position"] == "Attaquant" and random.random() < 0.5:
                                buts = min(random.randint(1, 2), buts_restants)
                            elif joueur["Position"] == "Milieu" and random.random() < 0.2:
                                buts = min(1, buts_restants)
                            elif joueur["Position"] == "Défenseur" and random.random() < 0.05:
                                buts = min(1, buts_restants)
                            buts_restants -= buts

                    data_matchs.append({
                        "ID_Match": id_match_counter,
                        "ID_Joueur": joueur["ID_Joueur"],
                        "Saison": saison,
                        "Date_Match": date_match,
                        "Catégorie": categorie,
                        "Compétition": competition,
                        "Titulaire": joueur in titulaires,
                        "Temps_Jeu": joueur["Temps_Jeu"],
                        "Carton_Jaune": carton_jaune,
                        "Carton_Rouge": carton_rouge,
                        "Fautes_Commisses": fautes_commisses,
                        "Buts": buts,
                    })
                
                buts_adversaire = random.randint(0, 5)
                score = f"{total_buts_match}-{buts_adversaire}"
                resultat = "Victoire" if total_buts_match > buts_adversaire else "Nul" if total_buts_match == buts_adversaire else "Défaite"
                
                for match in data_matchs:
                    if match["ID_Match"] == id_match_counter:
                        match["Score"] = score
                        match["Résultat"] = resultat
                
                id_match_counter += 1
            except ValueError as e:
                print(f"Erreur pour la catégorie {categorie} : {e}")

# Convertir en DataFrame
df_matchs = pd.DataFrame(data_matchs)

In [52]:
# Exporter en CSV
df_matchs.to_csv("matchs.csv", index=False)

In [53]:
import pandas as pd

# Charger les matchs
df_matchs = pd.read_csv("matchs.csv")  # Contient ID_Match, Date_Match, Résultat

# Ajouter une clé primaire séquentielle
df_matchs["ID_Match_Unique"] = range(1, len(df_matchs) + 1)

# Réorganiser les colonnes pour mettre ID_Match_Unique en premier
cols = ["ID_Match_Unique"] + [col for col in df_matchs.columns if col != "ID_Match_Unique"]
df_matchs = df_matchs[cols]



In [54]:
# Sauvegarder la table mise à jour
df_matchs.to_csv("matchs.csv", index=False)

print("✅ Clé primaire ajoutée avec succès : ID_Match_Unique")


✅ Clé primaire ajoutée avec succès : ID_Match_Unique


# Générer des données pour la table Données Athlétiques

In [56]:
import pandas as pd
# Chargement des données Joueurs et Matchs (à adapter selon votre source)
df_joueurs = pd.read_csv("joueurs.csv")  # Fichier contenant ID_Joueur, Catégorie, Poste
df_matchs = pd.read_csv("matchs.csv")  # Fichier contenant ID_Match, ID_Joueur, Temps_Jeu


In [57]:
# Définition des contraintes par catégorie
CONTRAINTES_CATEGORIE = {
    "Minime A": {"VO2_Max": (45, 55), "Distance_Parcourue": (6, 9), "Vitesse_Max": (23, 30), "Vitesse_Moyenne": (8, 12),
                 "Hauteur_Saut": (40, 60), "Puissance_Membres": (300, 600), "Force_Maximale": (50, 100)},
    "Minime B": {"VO2_Max": (45, 55), "Distance_Parcourue": (6, 9), "Vitesse_Max": (23, 30), "Vitesse_Moyenne": (8, 12),
                 "Hauteur_Saut": (40, 60), "Puissance_Membres": (300, 600), "Force_Maximale": (50, 100)},
    "Cadet A": {"VO2_Max": (48, 58), "Distance_Parcourue": (7, 10), "Vitesse_Max": (25, 32), "Vitesse_Moyenne": (9, 13),
                "Hauteur_Saut": (45, 65), "Puissance_Membres": (400, 700), "Force_Maximale": (60, 120)},
    "Cadet B": {"VO2_Max": (48, 58), "Distance_Parcourue": (7, 10), "Vitesse_Max": (25, 32), "Vitesse_Moyenne": (9, 13),
                "Hauteur_Saut": (45, 65), "Puissance_Membres": (400, 700), "Force_Maximale": (60, 120)},
    "Junior": {"VO2_Max": (50, 60), "Distance_Parcourue": (8, 11), "Vitesse_Max": (26, 34), "Vitesse_Moyenne": (10, 14),
               "Hauteur_Saut": (50, 70), "Puissance_Membres": (500, 800), "Force_Maximale": (70, 150)},
    "Elite": {"VO2_Max": (50, 60), "Distance_Parcourue": (8, 11), "Vitesse_Max": (26, 34), "Vitesse_Moyenne": (10, 14),
              "Hauteur_Saut": (50, 70), "Puissance_Membres": (500, 800), "Force_Maximale": (70, 150)},
    "Senior": {"VO2_Max": (52, 62), "Distance_Parcourue": (9, 12), "Vitesse_Max": (28, 36), "Vitesse_Moyenne": (11, 15),
               "Hauteur_Saut": (55, 80), "Puissance_Membres": (600, 1000), "Force_Maximale": (80, 200)}
}

In [58]:
# Ajustements par poste
AJUSTEMENTS_POSTE = {
    "Défenseur": {"Vitesse_Max": -2, "Vitesse_Moyenne": -1, "Distance_Parcourue": -1},
    "Gardien": {"Vitesse_Max": -5, "Vitesse_Moyenne": -3, "Distance_Parcourue": -3, "Hauteur_Saut": +10},
    "Milieu": {},  # Pas de modification
    "Attaquant": {"Vitesse_Max": +1, "Puissance_Membres": +50}
}

In [59]:
# Génération des données athlétiques
data_athletiques = []
ID_Athletique = 1  # Identifiant séquentiel

for match in df_matchs.itertuples():
    joueur = df_joueurs[df_joueurs["ID_Joueur"] == match.ID_Joueur].iloc[0]
    categorie = joueur["Catégorie"]
    poste = joueur["Position"]
    temps_jeu = match.Temps_Jeu

    contraintes = CONTRAINTES_CATEGORIE[categorie]
    ajustements = AJUSTEMENTS_POSTE.get(poste, {})

    vo2_max = round(random.uniform(*contraintes["VO2_Max"]), 2) + ajustements.get("VO2_Max", 0)
    distance_parcourue = max(0, round(random.uniform(*contraintes["Distance_Parcourue"]), 2) + ajustements.get("Distance_Parcourue", 0))
    vitesse_max = max(0, round(random.uniform(*contraintes["Vitesse_Max"]), 2) + ajustements.get("Vitesse_Max", 0))
    vitesse_moyenne = max(0, round(random.uniform(*contraintes["Vitesse_Moyenne"]), 2) + ajustements.get("Vitesse_Moyenne", 0))
    hauteur_saut = max(0, round(random.uniform(*contraintes["Hauteur_Saut"]), 2) + ajustements.get("Hauteur_Saut", 0))
    puissance_membres = max(0, round(random.uniform(*contraintes["Puissance_Membres"]), 2) + ajustements.get("Puissance_Membres", 0))
    force_maximale = max(0, round(random.uniform(*contraintes["Force_Maximale"]), 2) + ajustements.get("Force_Maximale", 0))

    # Si Temps_Jeu == 0, toutes les performances sont nulles
    if temps_jeu == 0:
        vo2_max = 0
        distance_parcourue = 0
        vitesse_max = 0
        vitesse_moyenne = 0
        hauteur_saut = 0
        puissance_membres = 0
        force_maximale = 0

    # Temps de récupération basé sur le temps de jeu (ex: 90 min => 10 min de récup)
    temps_recuperation = max(2, round((temps_jeu / 9), 2))

    data_athletiques.append({
        "ID_Athletique": ID_Athletique,
        "ID_Match": match.ID_Match,
        "ID_Joueur": match.ID_Joueur,
        "VO2_Max": vo2_max,
        "Distance_Parcourue": distance_parcourue,
        "Vitesse_Max": vitesse_max,
        "Vitesse_Moyenne": vitesse_moyenne,
        "Hauteur_Saut": hauteur_saut,
        "Puissance_Membres": puissance_membres,
        "Force_Maximale": force_maximale,
        "Temps_Récupération": temps_recuperation
    })
    ID_Athletique += 1

# Convertir en DataFrame
df_athletiques = pd.DataFrame(data_athletiques)

In [60]:
# Sauvegarde du DataFrame
df_athletiques.to_csv("donnees_athletiques.csv", index=False)

# Générer des données pour la table Données Techniques

In [61]:
import random
import pandas as pd
from faker import Faker

fake = Faker()


In [62]:
# Charger les données des joueurs
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur, Position, Temps_Jeu et ID_Match
df_matchs = pd.read_csv("matchs.csv")    # Contient ID_Match

In [63]:
data_techniques = []

In [64]:
position_constraints = {
    "Gardien": {
        "Nombre_Passes": (10, 30), "Précision_Passes": (85, 100),
        "Réception_Réussie": (5, 15), "Contrôle_Balle": (70, 90),
        "Nombre_Tirs": (0, 0), "Précision_Tirs": (0, 0),
        "Duels_Aériens_Gagnés": (5, 15), "Centres_Réussis": (0, 5),
        "Dribbles_Réussis": (0, 2), "Interceptions": (5, 15),
        "Tacles_Réussis": (2, 8), "Conversion_Occasions": (0, 0)
    },
    "Défenseur": {
        "Nombre_Passes": (30, 80), "Précision_Passes": (75, 95),
        "Réception_Réussie": (10, 25), "Contrôle_Balle": (75, 95),
        "Nombre_Tirs": (0, 3), "Précision_Tirs": (10, 40),
        "Duels_Aériens_Gagnés": (5, 20), "Centres_Réussis": (2, 10),
        "Dribbles_Réussis": (1, 6), "Interceptions": (10, 20),
        "Tacles_Réussis": (8, 20), "Conversion_Occasions": (2, 10)
    },
    "Milieu": {
        "Nombre_Passes": (50, 100), "Précision_Passes": (80, 98),
        "Réception_Réussie": (15, 30), "Contrôle_Balle": (85, 100),
        "Nombre_Tirs": (1, 5), "Précision_Tirs": (20, 60),
        "Duels_Aériens_Gagnés": (3, 10), "Centres_Réussis": (3, 15),
        "Dribbles_Réussis": (5, 12), "Interceptions": (8, 15),
        "Tacles_Réussis": (5, 15), "Conversion_Occasions": (5, 15)
    },
    "Attaquant": {
        "Nombre_Passes": (10, 40), "Précision_Passes": (70, 90),
        "Réception_Réussie": (10, 25), "Contrôle_Balle": (80, 100),
        "Nombre_Tirs": (5, 15), "Précision_Tirs": (40, 80),
        "Duels_Aériens_Gagnés": (3, 10), "Centres_Réussis": (5, 20),
        "Dribbles_Réussis": (5, 20), "Interceptions": (1, 5),
        "Tacles_Réussis": (0, 5), "Conversion_Occasions": (15, 40)
    }
}

In [65]:
ID_Technique = 1

for match in df_matchs.itertuples():
    joueur = df_joueurs[df_joueurs["ID_Joueur"] == match.ID_Joueur].iloc[0]
    poste = joueur["Position"]
    contraintes = position_constraints[poste]
    buts = match.Buts
    resultat = match.Résultat
    
    if match.Temps_Jeu == 0:
        valeurs = [0] * 12
    else:
        nombre_tirs = random.randint(*contraintes["Nombre_Tirs"])
        precision_tirs = round(random.uniform(*contraintes["Précision_Tirs"]), 2)
        conversion_occasions = (buts / nombre_tirs * 100) if nombre_tirs > 0 else 0
        
        # Ajustements en fonction du résultat
        impact = 1.1 if resultat == "Victoire" else 0.9 if resultat == "Défaite" else 1.0
        
        valeurs = [
            int(random.randint(*contraintes["Nombre_Passes"]) * impact),
            round(random.uniform(*contraintes["Précision_Passes"]) * impact, 2),
            int(random.randint(*contraintes["Réception_Réussie"]) * impact),
            round(random.uniform(*contraintes["Contrôle_Balle"]) * impact, 2),
            nombre_tirs,
            precision_tirs,
            int(random.randint(*contraintes["Duels_Aériens_Gagnés"]) * impact),
            int(random.randint(*contraintes["Centres_Réussis"]) * impact),
            int(random.randint(*contraintes["Dribbles_Réussis"]) * impact),
            int(random.randint(*contraintes["Interceptions"]) * impact),
            int(random.randint(*contraintes["Tacles_Réussis"]) * impact),
            round(conversion_occasions, 2)
        ]
    
    data_techniques.append([ID_Technique, match.ID_Match, match.ID_Joueur] + valeurs)
    ID_Technique += 1



In [66]:
# Conversion en DataFrame et sauvegarde
df_techniques = pd.DataFrame(data_techniques, columns=[
    "ID_Technique", "ID_Match", "ID_Joueur", "Nombre_Passes", "Précision_Passes", 
    "Réception_Réussie", "Contrôle_Balle", "Nombre_Tirs", "Précision_Tirs", 
    "Duels_Aériens_Gagnés", "Centres_Réussis", "Dribbles_Réussis", "Interceptions", 
    "Tacles_Réussis", "Conversion_Occasions"
])

In [67]:
# Exporter en CSV
df_techniques.to_csv("donnees_techniques.csv", index=False)

# Générer des données pour la table Données Tactiques

In [68]:
import random
import pandas as pd
from faker import Faker

fake = Faker()

In [69]:
# Charger les données des joueurs et des matchs
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur et Position
df_matchs = pd.read_csv("matchs.csv")    # Contient ID_Match, ID_Joueur, Temps_Jeu

In [70]:
# Définition des contraintes par position
position_constraints = {
    "Gardien": {
        "Transitions_Rapides": (0, 0), "Pressions_Réussies": (0, 0)
    },
    "Défenseur": {
        "Transitions_Rapides": (3, 8), "Pressions_Réussies": (5, 15)
    },
    "Milieu": {
        "Transitions_Rapides": (5, 12), "Pressions_Réussies": (10, 20)
    },
    "Attaquant": {
        "Transitions_Rapides": (8, 15), "Pressions_Réussies": (3, 10)
    }
}


In [71]:
# Zones d'influence possibles
zones_influence = ["Défensive", "Médiane", "Offensive", "Toute Surface"]

In [72]:
data_tactiques = []
# Générer les données tactiques
id_tactique = 1  # Initialiser ID_Tactique séquentiel

for match in df_matchs.itertuples():
    joueur = df_joueurs[df_joueurs["ID_Joueur"] == match.ID_Joueur].iloc[0]
    poste = joueur["Position"]
    contraintes = position_constraints[poste]
    resultat = match.Résultat
    
    # Impact du résultat du match
    impact = 1.1 if resultat == "Victoire" else 0.9 if resultat == "Défaite" else 1.0
    
    
    if match.Temps_Jeu == 0:
        valeurs = ["Aucune", 0, 0]
    else:
        valeurs = [
            "Défensive" if poste == "Gardien" else random.choice(zones_influence),
            int(random.randint(*contraintes["Transitions_Rapides"]) * impact),
            int(random.randint(*contraintes["Pressions_Réussies"]) * impact)
        ]
    
    data_tactiques.append([id_tactique, match.ID_Match, match.ID_Joueur] + valeurs)
    id_tactique += 1

# Conversion en DataFrame et sauvegarde
df_tactiques = pd.DataFrame(data_tactiques, columns=[
    "ID_Tactique", "ID_Match", "ID_Joueur", "Zones_Influence", "Transitions_Rapides", "Pressions_Réussies"
])

In [73]:
# Exporter en CSV
df_tactiques.to_csv("donnees_tactiques.csv", index=False)

# Générer des données pour la table Données Psychologiques

In [74]:
import random
import pandas as pd
from faker import Faker

fake = Faker()

In [75]:
# Charger les données des joueurs et des matchs
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur et Position
df_matchs = pd.read_csv("matchs.csv")    # Contient ID_Match, ID_Joueur, Temps_Jeu, Titulaire


In [76]:
data_psychologiques = []

# Générer les données psychologiques
id_psychologique = 1  # Initialiser ID_Psychologique séquentiel

for match in df_matchs.itertuples():
    joueur = df_joueurs[df_joueurs["ID_Joueur"] == match.ID_Joueur].iloc[0]
    temps_jeu = match.Temps_Jeu
    titulaire = match.Titulaire
    carton_jaune = match.Carton_Jaune
    carton_rouge = match.Carton_Rouge
    buts = match.Buts
    resultat = match.Résultat

    if temps_jeu == 0:
        confiance = 1  # Très faible
        resilience = random.randint(1, 3)
    elif 0 < temps_jeu < 45:
        confiance = random.randint(4, 6)  # Moyenne
        resilience = random.randint(5, 8)
    else:
        confiance = random.randint(7, 9)  # Très élevée
        resilience = random.randint(7, 9)
    
    # Appliquer les modifications en fonction des événements du match
    confiance += buts  # +1 par but marqué
    confiance -= carton_jaune  # -1 par carton jaune
    resilience -= carton_jaune  # -1 par carton jaune
    confiance -= carton_rouge * 3  # -3 par carton rouge
    resilience -= carton_rouge * 3  # -3 par carton rouge
    
    # Appliquer l'impact du résultat du match
    if resultat == "Victoire":
        confiance += 1
        resilience += 1
    elif resultat == "Défaite":
        confiance -= 1
        resilience -= 1
    
    # S'assurer que les valeurs restent dans la plage [1, 10]
    confiance = max(1, min(10, confiance))
    resilience = max(1, min(10, resilience))
    
    data_psychologiques.append([id_psychologique, match.ID_Match, match.ID_Joueur, confiance, resilience])
    id_psychologique += 1

# Conversion en DataFrame et sauvegarde
df_psychologiques = pd.DataFrame(data_psychologiques, columns=[
    "ID_Psychologique", "ID_Match", "ID_Joueur", "Confiance", "Résilience"
])

In [77]:
# Exporter en CSV
df_psychologiques.to_csv("donnees_psychologiques.csv", index=False)

# Générer des données pour la table Données Contextuelles

In [78]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime

fake = Faker()

In [79]:
# Charger les données des matchs
df_matchs_context = pd.read_csv("matchs.csv")  # Contient ID_Match, Date_Match, Résultat, Catégorie


In [80]:
# Supprimer les doublons pour ne garder qu'un seul ID_Match unique
df_matchs_context = df_matchs_context.drop_duplicates(subset=["ID_Match"])

In [81]:
# Définition des conditions saisonnières
saisons = {
    "Hiver": {"mois": [12, 1, 2], "météo": ["Pluvieux", "Nuageux", "Ensoleillé", "Venteux"], "terrain": ["Humide", "Boueux", "Sec"]},
    "Printemps": {"mois": [3, 4, 5], "météo": ["Ensoleillé", "Nuageux", "Pluvieux"], "terrain": ["Sec", "Humide"]},
    "Été": {"mois": [6, 7, 8], "météo": ["Ensoleillé", "Nuageux", "Venteux"], "terrain": ["Sec"]},
    "Automne": {"mois": [9, 10, 11], "météo": ["Nuageux", "Pluvieux", "Venteux", "Ensoleillé"], "terrain": ["Humide", "Boueux", "Sec"]}
}

In [82]:
# Générer une liste de 15 adversaires par catégorie pour la Ligue
ligue_adversaires = {categorie: random.sample([fake.last_name() + " FC" for _ in range(15)], 15)
                     for categorie in df_matchs_context["Catégorie"].unique()}

In [83]:
# Dictionnaire pour suivre la progression en Coupe par catégorie
coupe_progression = {categorie: 0 for categorie in df_matchs_context["Catégorie"].unique()}


In [84]:

data_contextuelles = []
id_contexte = 1  # Initialiser ID_Contexte séquentiel

for match in df_matchs_context.itertuples():
    date_match = datetime.strptime(match.Date_Match, "%Y-%m-%d")
    mois = date_match.month
    categorie = match.Catégorie
    saison = match.Saison  # Récupération directe de la saison
    competition = match.Compétition  # Utilisation directe

    # Déterminer la saison
    for saison, details in saisons.items():
        if mois in details["mois"]:
            # Générer dynamiquement les poids en fonction du nombre de choix disponibles
            nb_meteo = len(details["météo"])
            nb_terrain = len(details["terrain"])

            # Définir des poids proportionnels en fonction du nombre de choix
            meteo_weights = [0.5, 0.3, 0.15, 0.05][:nb_meteo]  # Ajuste pour éviter l'erreur
            terrain_weights = [0.6, 0.3, 0.1][:nb_terrain]  # Ajuste aussi pour éviter l'erreur

            # Choisir une météo et un état du terrain
            meteo = random.choices(details["météo"], weights=meteo_weights)[0]
            etat_terrain = random.choices(details["terrain"], weights=terrain_weights)[0]
            break

    # Attribution de l'adversaire en fonction de la compétition
    if competition == "Ligue":
        adversaire = random.choice(ligue_adversaires[categorie])

    elif competition == "Coupe":
        adversaire = fake.last_name() + " FC"
        if random.random() < 0.5:  # Simule un match gagné ou perdu en Coupe
            coupe_progression[categorie] += 1
        else:
            coupe_progression[categorie] = 5  # Élimination définitive

    elif competition == "Amical":
        adversaire = fake.last_name() + " FC"

    # Ajouter les données générées
    data_contextuelles.append({
        "ID_Contexte": id_contexte,
        "ID_Match": match.ID_Match,
        "Saison": saison,  # Ajout de la saison
        "Météo": meteo,
        "Etat_Terrain": etat_terrain,
        "Adversaire": adversaire,
        "Compétition": competition
    })
    id_contexte += 1  # Incrémenter l'ID séquentiel

# Convertir en DataFrame et sauvegarder
df_contextuelles = pd.DataFrame(data_contextuelles)

In [85]:
# Exporter en CSV
df_contextuelles.to_csv("donnees_contextuelles.csv", index=False)

# Générer des données pour la table Examen_Général

In [93]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

In [94]:
# Charger les joueurs avec leurs catégories
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur, Catégorie


In [95]:
# Définir les saisons sportives
saisons = [(2022, 2023), (2023, 2024), (2024, 2025)]


In [96]:
# Définir les paramètres médicaux en fonction des catégories (âges)
parametres_medecine = {
    "Minime B": {"temp": (36.5, 37.5), "glycémie": (70, 100), "tension": (100, 120)},
    "Minime A": {"temp": (36.5, 37.5), "glycémie": (70, 100), "tension": (100, 120)},
    "Cadet B": {"temp": (36.5, 37.8), "glycémie": (75, 105), "tension": (105, 125)},
    "Cadet A": {"temp": (36.5, 37.8), "glycémie": (75, 105), "tension": (105, 125)},
    "Junior": {"temp": (36.5, 38), "glycémie": (80, 110), "tension": (110, 130)},
    "Elite": {"temp": (36.5, 38), "glycémie": (80, 110), "tension": (110, 130)},
    "Senior": {"temp": (36.5, 38), "glycémie": (85, 120), "tension": (120, 140)},
}

In [97]:
# Génération des examens médicaux
data_examen_general = []
id_general = 1  # ID séquentiel

for saison in saisons:
    debut_saison = datetime(saison[0], 8, 15)  # Début saison en août
    fin_saison = datetime(saison[1], 6, 30)  # Fin saison en juin

    for joueur in df_joueurs.itertuples():
        categorie = joueur.Catégorie
        nb_examens = random.randint(1, 3)  # Chaque joueur passe entre 1 et 3 examens par saison

        for _ in range(nb_examens):
            date_examen = fake.date_between(start_date=debut_saison, end_date=fin_saison)
            
            # Récupérer les valeurs médicales en fonction de la catégorie
            parametres = parametres_medecine[categorie]
            temperature = round(random.uniform(*parametres["temp"]), 1)
            glycemie = round(random.uniform(*parametres["glycémie"]), 2)
            
            tension_min, tension_max = parametres["tension"]
            tension = f"{random.randint(tension_min, tension_max)}/{random.randint(60, 90)}"


            data_examen_general.append({
                "ID_General": id_general,
                "ID_Joueur": joueur.ID_Joueur,
                "Date_Examen": date_examen.strftime("%Y-%m-%d"),
                "Température": temperature,
                "Tension": tension,
                "Glycémie": glycemie,
                "Diabète_Type_1": random.choices([0, 1], weights=[0.98, 0.02])[0],  # Très rare
                "Diabète_Type_2": random.choices([0, 1], weights=[0.95, 0.05])[0],  # Plus fréquent chez les seniors
                "Asthme": random.choices([0, 1], weights=[0.9, 0.1])[0],  # Rare
                "Hypertension": random.choices([0, 1], weights=[0.8, 0.2 if categorie == "Senior" else 0.05])[0],
                "Allergies_Sévères": random.choices([0, 1], weights=[0.85, 0.15])[0]
            })

            id_general += 1  # Incrémentation ID séquentiel

# Convertir en DataFrame et sauvegarde
df_examen_general = pd.DataFrame(data_examen_general)

In [98]:
# Exporter en CSV
df_examen_general.to_csv("examen_general.csv", index=False)

# Générer des données pour la table Examen_Cardiopulmonaire

In [100]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

In [101]:
# Charger les joueurs avec leurs catégories
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur, Catégorie

# Définir les saisons sportives
saisons = [(2022, 2023), (2023, 2024), (2024, 2025)]

In [102]:
# Paramètres médicaux selon la catégorie (âge)
parametres_cardio = {
    "Minime B": {"freq_cardiaque": (70, 100), "capacite_pulmonaire": (3.0, 4.5), "test_effort": (3, 6)},
    "Minime A": {"freq_cardiaque": (70, 100), "capacite_pulmonaire": (3.0, 4.5), "test_effort": (3, 6)},
    "Cadet B": {"freq_cardiaque": (65, 95), "capacite_pulmonaire": (3.5, 5.0), "test_effort": (4, 7)},
    "Cadet A": {"freq_cardiaque": (65, 95), "capacite_pulmonaire": (3.5, 5.0), "test_effort": (4, 7)},
    "Junior": {"freq_cardiaque": (60, 90), "capacite_pulmonaire": (4.0, 5.5), "test_effort": (5, 8)},
    "Elite": {"freq_cardiaque": (55, 85), "capacite_pulmonaire": (4.5, 6.0), "test_effort": (6, 9)},
    "Senior": {"freq_cardiaque": (55, 80), "capacite_pulmonaire": (4.5, 6.5), "test_effort": (7, 10)},
}

In [103]:
# Génération des examens cardiopulmonaires
data_examen_cardio = []
id_cardio = 1  # ID séquentiel

for saison in saisons:
    debut_saison = datetime(saison[0], 8, 15)  # Début saison en août
    fin_saison = datetime(saison[1], 6, 30)  # Fin saison en juin

    for joueur in df_joueurs.itertuples():
        categorie = joueur.Catégorie
        nb_examens = random.randint(1, 2)  # Chaque joueur passe entre 1 et 2 examens par saison

        for _ in range(nb_examens):
            date_examen = fake.date_between(start_date=debut_saison, end_date=fin_saison)

            # Récupérer les paramètres selon la catégorie
            parametres = parametres_cardio[categorie]
            freq_cardiaque = random.randint(*parametres["freq_cardiaque"])
            capacite_pulmonaire = round(random.uniform(*parametres["capacite_pulmonaire"]), 1)
            test_effort = round(random.uniform(*parametres["test_effort"]), 2)

            data_examen_cardio.append({
                "ID_Cardio": id_cardio,
                "ID_Joueur": joueur.ID_Joueur,
                "Date_Examen": date_examen.strftime("%Y-%m-%d"),
                "Fréquence_Cardiaque": freq_cardiaque,  # En bpm
                "Capacité_Pulmonaire": capacite_pulmonaire,  # En litres
                "ECG_Anormal": random.choices([0, 1], weights=[0.95, 0.05])[0],  # ECG anormal rare (5%)
                "ETT_Anormal": random.choices([0, 1], weights=[0.97, 0.03])[0],  # ETT anormal très rare (3%)
                "Test_Effort": test_effort  # En minutes
            })

            id_cardio += 1  # Incrémentation ID séquentiel

# Convertir en DataFrame et sauvegarde
df_examen_cardio = pd.DataFrame(data_examen_cardio)

In [104]:
# Exporter en CSV
df_examen_cardio.to_csv("examen_cardiopulmonaire.csv", index=False)

# Générer des données pour la table Examen_Locomoteur

In [106]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

In [107]:
# Charger les joueurs avec leurs catégories
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur, Catégorie

# Définir les saisons sportives
saisons = [(2022, 2023), (2023, 2024), (2024, 2025)]

In [108]:
# Paramètres de blessures en fonction de l'âge (probabilités ajustées)
parametres_locomoteur = {
    "Minime B": {"lésion": 0.05, "douleurs": 0.1, "tendinite": 0.02, "déformation": 0.03, "opération": 0.01},
    "Minime A": {"lésion": 0.06, "douleurs": 0.12, "tendinite": 0.03, "déformation": 0.03, "opération": 0.02},
    "Cadet B": {"lésion": 0.1, "douleurs": 0.15, "tendinite": 0.05, "déformation": 0.05, "opération": 0.02},
    "Cadet A": {"lésion": 0.12, "douleurs": 0.18, "tendinite": 0.06, "déformation": 0.05, "opération": 0.03},
    "Junior": {"lésion": 0.15, "douleurs": 0.2, "tendinite": 0.1, "déformation": 0.05, "opération": 0.05},
    "Elite": {"lésion": 0.2, "douleurs": 0.25, "tendinite": 0.12, "déformation": 0.05, "opération": 0.07},
    "Senior": {"lésion": 0.25, "douleurs": 0.3, "tendinite": 0.15, "déformation": 0.08, "opération": 0.1},
}

In [109]:
# Génération des examens locomoteurs
data_examen_locomoteur = []
id_locomoteur = 1  # ID séquentiel

for saison in saisons:
    debut_saison = datetime(saison[0], 8, 15)  # Début saison en août
    fin_saison = datetime(saison[1], 6, 30)  # Fin saison en juin

    for joueur in df_joueurs.itertuples():
        categorie = joueur.Catégorie
        nb_examens = random.randint(1, 2)  # Chaque joueur passe entre 1 et 2 examens par saison

        for _ in range(nb_examens):
            date_examen = fake.date_between(start_date=debut_saison, end_date=fin_saison)

            # Récupérer les paramètres selon la catégorie
            parametres = parametres_locomoteur[categorie]
            
            data_examen_locomoteur.append({
                "ID_Locomoteur": id_locomoteur,
                "ID_Joueur": joueur.ID_Joueur,
                "Date_Examen": date_examen.strftime("%Y-%m-%d"),
                "Lésion_Musculaire": random.choices([0, 1], weights=[1 - parametres["lésion"], parametres["lésion"]])[0],
                "Douleurs_Articulaires": random.choices([0, 1], weights=[1 - parametres["douleurs"], parametres["douleurs"]])[0],
                "Tendinites_Chroniques": random.choices([0, 1], weights=[1 - parametres["tendinite"], parametres["tendinite"]])[0],
                "Déformation_Squelettique": random.choices([0, 1], weights=[1 - parametres["déformation"], parametres["déformation"]])[0],
                "Opération": random.choices([0, 1], weights=[1 - parametres["opération"], parametres["opération"]])[0]
            })

            id_locomoteur += 1  # Incrémentation ID séquentiel

# Convertir en DataFrame et sauvegarde
df_examen_locomoteur = pd.DataFrame(data_examen_locomoteur)

In [110]:
# Exporter en CSV
df_examen_locomoteur.to_csv("examen_locomoteur.csv", index=False)

# Générer des données pour la table Examen_ORL

In [111]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

In [112]:
# Charger les joueurs avec leurs catégories
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur, Catégorie

# Définir les saisons sportives
saisons = [(2022, 2023), (2023, 2024), (2024, 2025)]

In [113]:
# Paramètres ORL en fonction de l'âge (adaptés à la catégorie)
parametres_orl = {
    "Minime B": {"audition": (0, 10), "respiration": (4, 5), "gorge": (4, 5)},
    "Minime A": {"audition": (0, 12), "respiration": (3, 5), "gorge": (3, 5)},
    "Cadet B": {"audition": (0, 15), "respiration": (2, 5), "gorge": (2, 5)},
    "Cadet A": {"audition": (0, 18), "respiration": (2, 5), "gorge": (2, 5)},
    "Junior": {"audition": (0, 20), "respiration": (1, 5), "gorge": (1, 5)},
    "Elite": {"audition": (0, 25), "respiration": (1, 5), "gorge": (1, 5)},
    "Senior": {"audition": (0, 30), "respiration": (1, 5), "gorge": (1, 5)},
}

In [114]:
# Génération des examens ORL
data_examen_orl = []
id_orl = 1  # ID séquentiel

for saison in saisons:
    debut_saison = datetime(saison[0], 8, 15)  # Début saison en août
    fin_saison = datetime(saison[1], 6, 30)  # Fin saison en juin

    for joueur in df_joueurs.itertuples():
        categorie = joueur.Catégorie
        nb_examens = random.randint(1, 2)  # Chaque joueur passe entre 1 et 2 examens par saison

        for _ in range(nb_examens):
            date_examen = fake.date_between(start_date=debut_saison, end_date=fin_saison)

            # Récupérer les paramètres selon la catégorie
            parametres = parametres_orl[categorie]

            data_examen_orl.append({
                "ID_ORL": id_orl,
                "ID_Joueur": joueur.ID_Joueur,
                "Date_Examen": date_examen.strftime("%Y-%m-%d"),
                "Audition": round(random.uniform(*parametres["audition"]), 1),  # dB
                "Respiration_Nasale": random.randint(*parametres["respiration"]),  # Échelle de 1 à 5
                "État_Gorge": random.randint(*parametres["gorge"])  # Échelle de 1 à 5
            })

            id_orl += 1  # Incrémentation ID séquentiel

# Convertir en DataFrame et sauvegarde
df_examen_orl = pd.DataFrame(data_examen_orl)

In [115]:
# Exporter en CSV
df_examen_orl.to_csv("examen_orl.csv", index=False)

# Générer des données pour la table Examen_Stomatologique

In [116]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

In [117]:
# Charger les joueurs avec leurs catégories
df_joueurs = pd.read_csv("joueurs.csv")  # Contient ID_Joueur, Catégorie

# Définir les saisons sportives
saisons = [(2022, 2023), (2023, 2024), (2024, 2025)]

In [118]:
# Paramètres Stomatologiques en fonction de l'âge (adaptés à la catégorie)
parametres_stomato = {
    "Minime B": {"caries": (0, 3), "gencives": (3, 5), "mâchoire": (3, 5)},
    "Minime A": {"caries": (0, 3), "gencives": (3, 5), "mâchoire": (3, 5)},
    "Cadet B": {"caries": (0, 4), "gencives": (2, 5), "mâchoire": (2, 5)},
    "Cadet A": {"caries": (0, 4), "gencives": (2, 5), "mâchoire": (2, 5)},
    "Junior": {"caries": (0, 5), "gencives": (1, 5), "mâchoire": (1, 5)},
    "Elite": {"caries": (0, 5), "gencives": (1, 5), "mâchoire": (1, 5)},
    "Senior": {"caries": (0, 5), "gencives": (1, 5), "mâchoire": (1, 5)},
}

In [119]:
# Génération des examens Stomatologiques
data_examen_stomatologique = []
id_stomato = 1  # ID séquentiel

for saison in saisons:
    debut_saison = datetime(saison[0], 8, 15)  # Début saison en août
    fin_saison = datetime(saison[1], 6, 30)  # Fin saison en juin

    for joueur in df_joueurs.itertuples():
        categorie = joueur.Catégorie
        nb_examens = random.randint(1, 2)  # Chaque joueur passe entre 1 et 2 examens par saison

        for _ in range(nb_examens):
            date_examen = fake.date_between(start_date=debut_saison, end_date=fin_saison)

            # Récupérer les paramètres selon la catégorie
            parametres = parametres_stomato[categorie]

            data_examen_stomatologique.append({
                "ID_Stomatologique": id_stomato,
                "ID_Joueur": joueur.ID_Joueur,
                "Date_Examen": date_examen.strftime("%Y-%m-%d"),
                "Nombre_Caries": random.randint(*parametres["caries"]),
                "État_Gencives": random.randint(*parametres["gencives"]),  # Échelle de 1 à 5
                "Problèmes_Mâchoire": random.randint(*parametres["mâchoire"])  # Échelle de 1 à 5
            })

            id_stomato += 1  # Incrémentation ID séquentiel

# Convertir en DataFrame et sauvegarde
df_examen_stomatologique = pd.DataFrame(data_examen_stomatologique)

In [120]:
# Exporter en CSV
df_examen_stomatologique.to_csv("examen_stomatologique.csv", index=False)

# Generer la table objectifs joueurs

In [1]:
import pandas as pd
import random
from faker import Faker

In [2]:
# Charger les joueurs
df_joueurs = pd.read_csv("joueurs.csv")  

In [3]:
# Faker pour ajouter un peu d'aléatoire
fake = Faker()

In [4]:
# Définir les objectifs possibles par poste
objectifs_par_poste = {
    "Gardien": [
        ("Précision_Passes", random.uniform(70, 80), "%"),
        ("Minutes_Totales", random.randint(1500, 2200), "minutes"),
    ],
    "Défenseur": [
        ("Distance_Parcourue_Match", random.uniform(8, 9.5), "km"),
        ("Précision_Passes", random.uniform(80, 85), "%"),
        ("Tacles_Réussis", random.uniform(60, 75), "%"),
    ],
    "Milieu": [
        ("Distance_Parcourue_Match", random.uniform(9, 11), "km"),
        ("Précision_Passes", random.uniform(85, 90), "%"),
        ("Tirs_Cadrés", random.uniform(40, 60), "%"),
    ],
    "Attaquant": [
        ("Distance_Parcourue_Match", random.uniform(8.5, 10), "km"),
        ("Vitesse_Max", random.uniform(30, 34), "km/h"),
        ("Conversion_Tirs_Buts", random.uniform(30, 45), "%"),
    ]
}


In [5]:
# Définir les saisons (comme dans tes matchs)
saisons = ["2022-2023", "2023-2024", "2024-2025"]


In [6]:
# Générer les objectifs
objectifs = []
id_objectif = 1

for saison in saisons:
    for joueur in df_joueurs.itertuples():
        poste = joueur.Position  # Attention : il faut que la colonne s'appelle exactement 'Position' dans ton fichier CSV

        if poste not in objectifs_par_poste:
            continue  # Si jamais un poste est inconnu

        objectifs_pour_poste = objectifs_par_poste[poste]

        for objectif in objectifs_pour_poste:
            objectifs.append({
                "ID_Objectif": id_objectif,
                "ID_Joueur": joueur.ID_Joueur,
                "Saison": saison,
                "Objectif_Type": objectif[0],
                "Valeur_Cible": round(objectif[1], 2),
                "Unite": objectif[2]
            })
            id_objectif += 1
# Convertir en DataFrame
df_objectifs = pd.DataFrame(objectifs)

In [7]:
# Exporter le résultat
df_objectifs.to_csv("objectifs_joueur.csv", index=False)

print("✅ Fichier objectifs_joueur.csv généré avec succès !")

✅ Fichier objectifs_joueur.csv généré avec succès !


# Generer la table entrainement

In [8]:
import pandas as pd
import random
from datetime import datetime, timedelta

In [9]:
# Charger les joueurs et les matchs
df_joueurs = pd.read_csv("joueurs.csv")
df_matchs = pd.read_csv("matchs.csv")

In [10]:
# Préparer les dates de matchs par catégorie
matchs_par_categorie = df_matchs.groupby("Catégorie")["Date_Match"].apply(list).to_dict()

In [11]:
# Saisons disponibles
saisons = {
    "2022-2023": (datetime(2022, 8, 21), datetime(2023, 6, 30)),
    "2023-2024": (datetime(2023, 8, 21), datetime(2024, 6, 30)),
    "2024-2025": (datetime(2024, 8, 21), datetime(2025, 6, 30)),
}

In [12]:
# Types d'exercices
types_exercices = ["Endurance", "Sprint", "Technique", "Match d’entraînement"]

# Intensités
intensites = ["Faible", "Moyenne", "Élevée"]

In [13]:
# Fonction pour vérifier jour de match
def est_jour_match(categorie, date):
    dates_matchs = matchs_par_categorie.get(categorie, [])
    return date.strftime("%Y-%m-%d") in dates_matchs

In [14]:
# Simulation des entraînements
data_entrainements = []
id_entrainement = 1

for saison, (debut_saison, fin_saison) in saisons.items():
    debut_entrainement = debut_saison - timedelta(days=30)

    semaine_courante = debut_entrainement
    while semaine_courante <= fin_saison:
        jours_semaine = [semaine_courante + timedelta(days=i) for i in range(7)]

        for jour in jours_semaine:
            if jour > fin_saison:
                break

            jour_semaine = jour.weekday()
            if jour_semaine in [0, 1, 2, 3, 4]:  # Lundi-vendredi uniquement
                for categorie, joueurs_categorie in df_joueurs.groupby("Catégorie"):
                    if est_jour_match(categorie, jour):
                        continue  # Pas d'entraînement pour cette catégorie ce jour

                    joueurs_presents = joueurs_categorie.sample(frac=random.uniform(0.8, 0.9))

                    for joueur in joueurs_presents.itertuples():
                        duree = random.randint(60, 120)
                        intensite = random.choice(intensites)

                        if intensite == "Faible":
                            distance = round(random.uniform(3, 5), 2)
                            sprint_max = round(random.uniform(20, 26), 2)
                        elif intensite == "Moyenne":
                            distance = round(random.uniform(5, 7.5), 2)
                            sprint_max = round(random.uniform(26, 30), 2)
                        else:
                            distance = round(random.uniform(7.5, 10), 2)
                            sprint_max = round(random.uniform(30, 33), 2)

                        type_exercice = random.choice(types_exercices)

                        data_entrainements.append({
                            "ID_Entrainement": id_entrainement,
                            "ID_Joueur": joueur.ID_Joueur,
                            "Date_Entrainement": jour.strftime("%Y-%m-%d"),
                            "Durée": duree,
                            "Intensité": intensite,
                            "Distance_Parcourue": distance,
                            "Sprint_Max": sprint_max,
                            "Type_Exercice": type_exercice,
                            "Catégorie": categorie,
                            "Saison": saison
                        })
                        id_entrainement += 1
        semaine_courante += timedelta(days=7)

# Convertir en DataFrame
df_entrainements = pd.DataFrame(data_entrainements)

In [15]:
# Exporter
df_entrainements.to_csv("entrainements.csv", index=False)

print("✅ Fichier entrainements.csv généré avec succès !")

✅ Fichier entrainements.csv généré avec succès !
